In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
data_augment = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/data_augment.xlsx')

In [ ]:
fontPath = '/content/drive/MyDrive/image/times.ttf'
font_manager.fontManager.addfont(fontPath)

In [ ]:
font = FontProperties()
font.set_family('Times New Roman')
font.set_size(20)

In [ ]:
!cp /content/drive/MyDrive/image/images.zip -d /content/images.zip
!unzip -q images.zip

In [ ]:
!rm /content/images.zip

In [ ]:
import os
A=os.listdir('/content')
A

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
s='/content/images/*'
path=glob.glob(s)
import re
def sorted_alphanumeric(data):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ]
    return sorted(data, key=alphanum_key)

path1=sorted_alphanumeric(path)
path1 = [item.replace('p','_') for item in path1]
data=pd.read_excel('/content/drive/MyDrive/image/data_c.xlsx')
data
len(path1)
print(path1)
type(path1)
path1=np.array(path1)
path1.shape
path1[0]

In [ ]:
path1=np.array(path1)
data=np.array(data)
data
len(data)

In [ ]:
df=pd.DataFrame({'imgpath':path1,'Porosity':data[:,0],'throat radius':data[:,1],'pore radius':data[:,2],'pore_connection_number':data[:,3],'pore shape factor':data[:,4]})

In [ ]:
df

In [ ]:
df_new=df[df['Porosity'] <0.35 ]
df_new.shape
df_new

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data2=np.array(df_new)
scaler = StandardScaler()
data1= scaler.fit_transform(data2[:,1:])
print(data1)
print(type(data1))
data1.shape

In [ ]:
data2

In [ ]:
df=pd.DataFrame({'imgpath':df_new['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

In [ ]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.15,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

In [ ]:
def Data_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img = []
                y_batch=[]
                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    Porosity=np.array(df.Porosity)[idd]
                    throat_radius=np.array(df['throat radius'])[idd]
                    pore_radius=np.array(df['pore radius'])[idd]
                    pore_connection_number=np.array(df.pore_connection_number)[idd]
                    pore_shape_factor=np.array(df['pore shape factor'])[idd]
                    y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
                    y=np.array([y_1])
                    x_batch_img.append(img_1)
                    y_batch.append(y)


                x_batch_img = np.array(x_batch_img)
                y_batch= np.array(y_batch)
                y_batch=y_batch.reshape(-1,5)
              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img , y_batch

In [ ]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img_2

In [ ]:
batch_size=10
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
print(nbatches_valid,nbatches_train,nbatches_test)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf

In [ ]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

In [ ]:
import os
checkpoint_path="/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_model_weights/cp-{epoch:03d}.ckpt"
#os.makedirs("/content/drive/MyDrive/image/porosity_filtering/training_5layers", exist_ok=True)
#ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/image/porosity_filtering/training_model_5layers/weights.{epoch:02d}-{val_loss:.2f}.hdf5', monitor='val_loss')
cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_193.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger])

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_model_weights/cp-142.ckpt'
model.load_weights(wieght)

In [ ]:
history2=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=141)

In [ ]:
history3=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=142)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_model_weights/cp-177.ckpt'
model.load_weights(wieght)

In [ ]:
history3=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=177)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_model_weights/cp-192.ckpt'
model.load_weights(wieght)

In [ ]:
history4=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=192)

In [ ]:
#**********************************************predicting***********************************************

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_model_weights/cp-184.ckpt'
model.load_weights(wieght)

In [ ]:
model.evaluate_generator(test_gen, nbatches_test, workers=1)

In [ ]:
y=model.predict_generator(
    test_gen_pre,
    verbose=1,
    steps=nbatches_test,
    callbacks=None,
    max_queue_size=10,
    workers=1,
    use_multiprocessing=False)

In [ ]:
y_1=pd.DataFrame({'porosity':((y[:,0]*df_new['Porosity'].std())+df_new['Porosity'].mean()),
                  'throat radius':((y[:,1]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((y[:,2]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((y[:,3]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((y[:,4]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
df_test_1=np.array(df_test)
df_test_2=pd.DataFrame({'porosity':((df_test_1[:,1]*df_new.Porosity.std())+df_new.Porosity.mean()),
                  'throat radius':((df_test_1[:,2]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((df_test_1[:,3]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((df_test_1[:,4]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((df_test_1[:,5]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_pore_shape_factor=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_shape_factor)

In [ ]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_throat_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_throat_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_pore_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_pore_connection_number=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_connection_number)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/Five_layers/training_model_weights/cp-200.ckpt'
model.load_weights(wieght)

In [ ]:
model.evaluate_generator(test_gen, nbatches_test, workers=1)

In [ ]:
y=model.predict_generator(
    test_gen_pre,
    verbose=1,
    steps=nbatches_test,
    callbacks=None,
    max_queue_size=10,
    workers=1,
    use_multiprocessing=False)

In [ ]:
y_1=pd.DataFrame({'porosity':((y[:,0]*df_new['Porosity'].std())+df_new['Porosity'].mean()),
                  'throat radius':((y[:,1]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((y[:,2]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((y[:,3]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((y[:,4]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
df_test_1=np.array(df_test)
df_test_2=pd.DataFrame({'porosity':((df_test_1[:,1]*df_new.Porosity.std())+df_new.Porosity.mean()),
                  'throat radius':((df_test_1[:,2]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((df_test_1[:,3]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((df_test_1[:,4]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((df_test_1[:,5]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_pore_shape_factor=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_shape_factor)

In [ ]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_throat_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_throat_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_pore_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_pore_connection_number=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_connection_number)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)
x_values_1 = np.array(df_test_2['porosity']).tolist()
y_values_1 = np.array(y_1['porosity']).tolist()
fig = plt.figure(figsize =(10, 7),dpi=200)
plt.scatter(x_values_1,y_values_1)
p1 = max(max(y_values_1), max(x_values_1))
p2 = min(min(y_values_1), min(x_values_1))
plt.plot([p1, p2], [p1, p2], 'b--')
plt.axis([p2-0.1, p1+0.1, p2-0.1, p1+0.1])
#linear regretion
x_values_1, y_values_1 = x_values.reshape(-1,1),y_values.reshape(-1,1)
reg = LinearRegression().fit(x_values_1, y_values_1)
#print(reg.get_params())
plt.plot(x_values_1,reg.predict(x_values_1),'k-')
plt.xlabel("porosity")
plt.ylabel("prediction")
plt.title('r_squared_porosity=0.99')
plt.show
fig.savefig('porosity_r_squared.png',dpi=200)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)
x_values_1 = np.array(df_test_2['pore shape factor']).tolist()
y_values_1 = np.array(y_1['pore shape factor']).tolist()
fig= plt.figure(figsize =(10, 7),dpi=200)
plt.scatter(x_values_1,y_values_1)
p1 = max(max(y_values_1), max(x_values_1))
p2 = min(min(y_values_1), min(x_values_1))
plt.plot([p1, p2], [p1, p2], 'b--')
plt.axis([p2-0.005, p1+0.005, p2-0.005, p1+0.005])
#linear regretion
x_values_1, y_values_1 = x_values.reshape(-1,1),y_values.reshape(-1,1)
reg = LinearRegression().fit(x_values_1, y_values_1)
#print(reg.get_params())
plt.plot(x_values_1,reg.predict(x_values_1),'k-')
plt.xlabel("pore shape factor")
plt.ylabel("prediction")
plt.show()
fig.savefig('pore shape factor_r_squared.jpg',dpi=200)


In [ ]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)
x_values_1 = np.array(df_test_2['throat radius']).tolist()
y_values_1 = np.array(y_1['throat radius']).tolist()
fig=plt.figure(figsize=(10,7),dpi=200)
plt.scatter(x_values_1,y_values_1)
p1 = max(max(y_values_1), max(x_values_1))
p2 = min(min(y_values_1), min(x_values_1))
plt.plot([p1, p2], [p1, p2], 'b--')
plt.axis([p2, p1, p2, p1])
#linear regretion
x_values_1, y_values_1 = x_values.reshape(-1,1),y_values.reshape(-1,1)
reg = LinearRegression().fit(x_values_1, y_values_1)
#print(reg.get_params())
plt.plot(x_values_1,reg.predict(x_values_1),'k-')
plt.xlabel('throat radius')
plt.ylabel("prediction")
plt.show()
fig.savefig('throat radius_r_squared.png',dpi=200)


In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)
x_values_1 = np.array(df_test_2['pore radius']).tolist()
y_values_1 = np.array(y_1['pore radius']).tolist()
fig=plt.figure(figsize=(10,7),dpi=200)
plt.scatter(x_values_1,y_values_1)
p1 = max(max(y_values_1), max(x_values_1))
p2 = min(min(y_values_1), min(x_values_1))
plt.plot([p1, p2], [p1, p2], 'b--')
plt.axis([p2, p1, p2, p1])
#linear regretion
x_values_1, y_values_1 = x_values.reshape(-1,1),y_values.reshape(-1,1)
reg = LinearRegression().fit(x_values_1, y_values_1)
#print(reg.get_params())
plt.plot(x_values_1,reg.predict(x_values_1),'k-')
plt.xlabel('pore radius')
plt.ylabel("prediction")
plt.show()
fig.savefig('pore radius_r_squared.png',dpi=200)


In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)
x_values_1 = np.array(df_test_2['pore_connection_number']).tolist()
y_values_1 = np.array(y_1['pore_connection_number']).tolist()
fig=plt.figure(figsize=(10,7),dpi=200)
plt.scatter(x_values_1,y_values_1)
p1 = max(max(y_values_1), max(x_values_1))
p2 = min(min(y_values_1), min(x_values_1))
plt.plot([p1, p2], [p1, p2], 'b--')
plt.axis([p2-0.1, p1+0.1, p2-0.1, p1+0.1])
#linear regretion
x_values_1, y_values_1 = x_values.reshape(-1,1),y_values.reshape(-1,1)
reg = LinearRegression().fit(x_values_1, y_values_1)
#print(reg.get_params())
plt.plot(x_values_1,reg.predict(x_values_1),'k-')
plt.xlabel('pore_connection_number')
plt.ylabel("prediction")
plt.show()
fig.savefig('pore_connection_number_r_squared.png',dpi=200)


In [ ]:
import pandas as pd
data_layers= pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/Five_layers/layers.xlsx')

In [ ]:
 data_layers.val_mse_6layers[0], data_layers.val_mse_6layers[16]=5,8


In [ ]:
data_layers.val_mse_5layers[199]=0.1003

In [ ]:
data_layers

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize =(12, 9),dpi=300)
plt.semilogy( data_layers.epoch,data_layers.val_mse_4layers,label='4_layers')
plt.xlabel('Training epochs', fontsize = 30, fontname = 'Times New Roman')
plt.ylabel('Validating MSE', fontsize = 30, fontname = 'Times New Roman')
plt.semilogy(data_layers.epoch,data_layers.val_mse_5layers,label='5_layers')
plt.semilogy(data_layers.epoch,data_layers.val_mse_6layers,label='6_layers')
plt.xticks(fontsize = 30, fontname = 'Times New Roman')
plt.yticks(fontsize = 30, fontname = 'Times New Roman')
plt.legend(fontsize = 30,loc='upper left',prop = font)
fig.savefig('layers.png',dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize =(15, 10),dpi=300)
ax.semilogy( data_layers.epoch[159:],data_layers.val_mse_4layers[159:],label='4_layers',lw=4)
ax.semilogy(data_layers.epoch[159:],data_layers.val_mse_5layers[159:],label='5_layers',lw=4)
ax.semilogy(data_layers.epoch[159:],data_layers.val_mse_6layers[159:],label='6_layers',lw=4)

ax.tick_params(axis = 'y', which='major', labelsize = 40)
ax.tick_params(axis = 'y',which= 'minor',labelsize = 40)
ax.tick_params(axis = 'x',labelsize = 40)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname = 'Times New Roman')
labels = ax.get_yticklabels(minor= True) + ax.get_yticklabels(minor= False)
for label in labels:
    label.set_fontname('Times New Roman')

for spine in ax.spines.values():
    spine.set_linewidth(2.5)

#plt.semilogy( data_layers.epoch[159:],data_layers.val_mse_4layers[159:],label='4_layers')
#plt.xlabel('epoch', fontsize = 20)
#plt.ylabel('loss', fontsize = 20)
#plt.semilogy(data_layers.epoch[159:],data_layers.val_mse_5layers[159:],label='5_layers')
#plt.semilogy(data_layers.epoch[159:],data_layers.val_mse_6layers[159:],label='6_layers')
#plt.xticks(fontsize = 20)
#plt.yticks(fontsize = 20)
#plt.legend(fontsize = 20)
fig.savefig('layers_cut.png',dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
x = ['4', '5', '6']
y = [5.1, 5.65, 5.79]
fig = plt.figure(figsize =(12, 9),dpi=300)
plt.bar([1,2,3], y,width=0.1)
plt.xticks([1,2,3], x,fontsize= 30,fontname = 'Times New Roman')
plt.yticks(fontsize= 30,fontname = 'Times New Roman')

plt.xlabel('Layer',fontsize = 30,fontname = 'Times New Roman')
plt.ylabel("Rumtime In Minutes",fontsize =30,fontname = 'Times New Roman')
fig.savefig('time_layer.png',dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
x = ['7-5-3-3-3','7-3-3-3-3','11-7-5-3-3','3-3-3-3-3','8-4-2-2-2','6-4-2-2-2']
y = [5.65, 4.33, 13.87, 2.8, 5.35 ,5.75]
fig = plt.figure(figsize =(10, 7),dpi=300)
plt.bar([1,2,3,4,5,6], y,width=0.2)
plt.xticks([1,2,3,4,5,6], x,fontsize= 20,fontname = 'Times New Roman')
plt.yticks(fontsize= 20,fontname = 'Times New Roman')
plt.xlabel('Kernel',fontsize = 20,fontname = 'Times New Roman')
plt.ylabel("Rumtime In Minutes",fontsize =20,fontname = 'Times New Roman')
fig.savefig('time_kernel.png',dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
x = ['0.001', '0.003', '0.0005','Sc2','Sc3']
y = [5.65, 5.45, 5.36, 5.42,5.22]
fig = plt.figure(figsize =(10, 7),dpi=200)
plt.bar([1,2,3,4,5], y)
plt.xticks([1,2,3,4,5], x,fontsize= 15)
plt.yticks(fontsize= 15)

plt.xlabel('learning Rate',fontsize = 20)
plt.ylabel("rumtime (min)",fontsize =20)
fig.savefig('/content/drive/MyDrive/image/porosity_filtering/Five_layers/time_learning_rate.png',dpi=200)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/filter.xlsx')
data

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=200)
plt.semilogy(data.epoch,data['val_mse_32_32'])
plt.semilogy(data.epoch,data['val_mse_512'])
plt.semilogy(data.epoch,data['val_mse_12'])
plt.semilogy(data.epoch,data['val_mse_16'])
plt.legend(['32-32-64-64-128','32-64-128-256-512','12-24-48-96-192','16-32-64-128-256'],prop = font,loc='upper left')
plt.xlabel('Training epochs',fontsize = 30,fontname = 'Times New Roman')
plt.ylabel('Validating MSE',fontsize = 30,fontname = 'Times New Roman')
plt.yticks(fontsize=30,fontname = 'Times New Roman')
plt.xticks(fontsize= 30,fontname = 'Times New Roman')
plt.show()
fig.savefig('filter_accuracy',dpi = 200)


In [ ]:
fig,ax = plt.subplots(figsize = (20,15), dpi=300)
ax.semilogy(data.epoch[159:],data['val_mse_32_32'][159:], lw = 5)
ax.semilogy(data.epoch[159:],data['val_mse_512'][159:], lw = 5)
ax.semilogy(data.epoch[159:],data['val_mse_12'][159:], lw = 5)
ax.semilogy(data.epoch[159:],data['val_mse_16'][159:], lw = 5)
ax.tick_params(axis = 'y', which='major', labelsize = 40)
ax.tick_params(axis = 'y',which= 'minor',labelsize = 40)
ax.tick_params(axis = 'x',labelsize = 40)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname = 'Times New Roman')
labels = ax.get_yticklabels(minor= True) + ax.get_yticklabels(minor= False)
for label in labels:
    label.set_fontname('Times New Roman')

for spine in ax.spines.values():
    spine.set_linewidth(2.5)



#plt.semilogy(data.epoch[159:],data['val_mse_32_32'][159:])
#plt.semilogy(data.epoch[159:],data['val_mse_512'][159:])
#plt.semilogy(data.epoch[159:],data['val_mse_12'][159:])
#plt.semilogy(data.epoch[159:],data['val_mse_16'][159:])
#plt.legend(['32-32-64-64-128','32-64-128-256-512','12-24-48-96-192','16-32-64-128-256'],fontsize = 20)
#plt.xlabel('epoch',fontsize = 20)
#plt.ylabel('loss',fontsize = 20)
#plt.yticks(fontsize= 20)
#lt.xticks(fontsize= 20)


plt.show()
fig.savefig('filter_accuracy_cut',dpi = 300)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
fig = plt.figure(figsize =(10,7),dpi = 200)
y = [5.2,7.56,10.02,5.65]
x = ['12-24-48-96-192','32-32-64-64-128','32-64-128-256-512','16-32-64-128-256']
plt.bar([1,2,3,4],y)
plt.xticks([1,2,3,4],x,fontsize=13.5)
plt.yticks(fontsize = 20)
plt.ylabel('Rumtime In Minutes',fontsize = 20)
plt.xlabel('Filter Numbers',fontsize = 20)
plt.savefig('filter_runtime.png',dpi = 200)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data_kernel = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/kernel.xlsx')
data_kernel

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=300)
plt.semilogy(data_kernel.epoch,data_kernel['7-5-3-3-3'])
plt.semilogy(data_kernel.epoch,data_kernel['7-3-3-3-3'])
plt.semilogy(data_kernel.epoch,data_kernel['11-7-5-3-3'])
plt.semilogy(data_kernel.epoch,data_kernel['3-3-3-3-3'])
plt.semilogy(data_kernel.epoch,data_kernel['6-4-2-2-2'])
plt.semilogy(data_kernel.epoch,data_kernel['8-4-2-2-2'])
plt.legend(['7-5-3-3-3','7-3-3-3-3','11-7-5-3-3','3-3-3-3-3','6-4-2-2-2','8-4-2-2-2'], prop = font , loc='upper left')
plt.xlabel('Training epochs', fontsize = 30, fontname = 'Times New Roman')
plt.ylabel('Validating MSE', fontsize = 30, fontname = 'Times New Roman')
plt.yticks(fontsize=30, fontname = 'Times New Roman')
plt.xticks(fontsize= 30,fontname = 'Times New Roman')
plt.show()
fig.savefig('kernel_accuracy',dpi = 300)


In [ ]:
fig, ax = plt.subplots(figsize = (15,10), dpi = 300)
ax.semilogy(data_kernel.epoch[159:],data_kernel['7-5-3-3-3'][159:],lw =4)
ax.semilogy(data_kernel.epoch[159:],data_kernel['7-3-3-3-3'][159:],lw=4)
ax.semilogy(data_kernel.epoch[159:],data_kernel['11-7-5-3-3'][159:],lw=4)
ax.semilogy(data_kernel.epoch[159:],data_kernel['3-3-3-3-3'][159:],lw=4)
ax.semilogy(data_kernel.epoch[159:],data_kernel['6-4-2-2-2'][159:],lw=4)
ax.semilogy(data_kernel.epoch[159:],data_kernel['8-4-2-2-2'][159:],lw=4)
ax.tick_params(axis = 'y', which = 'major', labelsize = 40)
ax.tick_params(axis = 'y',which = 'minor', labelsize = 40)
ax.tick_params(axis = 'x', labelsize = 40)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname ='Times New Roman')
labels = ax.get_yticklabels(minor = True) + ax.get_yticklabels(minor = False)
for label in labels:
    label.set_fontname('Times New Roman')

for spine in ax.spines.values():
    spine.set_linewidth(2.5)




#fig = plt.figure(figsize = (15,10),dpi=300)
#plt.semilogy(data_kernel.epoch[159:],data_kernel['7-5-3-3-3'][159:])
#plt.semilogy(data_kernel.epoch[159:],data_kernel['7-3-3-3-3'][159:])
#plt.semilogy(data_kernel.epoch[159:],data_kernel['11-7-5-3-3'][159:])
#plt.semilogy(data_kernel.epoch[159:],data_kernel['3-3-3-3-3'][159:])
#plt.semilogy(data_kernel.epoch[159:],data_kernel['6-4-2-2-2'][159:])
#plt.semilogy(data_kernel.epoch[159:],data_kernel['8-4-2-2-2'][159:])
#plt.legend(['7-5-3-3-3','7-3-3-3-3','11-7-5-3-3','3-3-3-3-3','6-4-2-2-2','8-4-2-2-2'],fontsize = 15,loc='upper left')
#plt.xlabel('epoch',fontsize = 20)
#plt.ylabel('loss',fontsize = 20)
#plt.yticks(fontsize=30)
#plt.xticks(fontsize= 20)
plt.show()
fig.savefig('kernel_accuracy_cut',dpi = 300)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data_learningrate = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/learning rates_revised.xlsx')

In [ ]:
data_learningrate

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=300)
plt.semilogy(data_learningrate.epoch,data_learningrate['val_loss_decay'])
plt.semilogy(data_learningrate.epoch,data_learningrate['val_loss_floor'])
plt.semilogy(data_learningrate.epoch,data_learningrate['val_loss_exp'])
plt.semilogy(data_learningrate.epoch,data_learningrate['val_loss_0.0005'])
plt.semilogy(data_learningrate.epoch,data_learningrate['val_mse_0.003'])
plt.semilogy(data_learningrate.epoch,data_learningrate['val_mse_0.001'])
plt.legend(['Sc1','Sc2','Sc3','0.0005','0.003','0.001'],prop = font ,loc='upper left')
plt.xlabel('Training epochs',fontsize = 30, fontname = 'Times New Roman')
plt.ylabel('Validating MSE',fontsize = 30, fontname = 'Times New Roman')
plt.yticks(fontsize=30,fontname = 'Times New Roman')
plt.xticks(fontsize= 30,fontname = 'Times New Roman')
plt.show()
fig.savefig('learning rate_accuracy',dpi = 300)


In [ ]:
fig,ax = plt.subplots(figsize = (20,15), dpi = 300)
ax.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_decay'][159:])
ax.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_floor'][159:])
ax.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_exp'][159:])
ax.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_0.0005'][159:])
ax.semilogy(data_learningrate.epoch[159:],data_learningrate['val_mse_0.003'][159:])
ax.semilogy(data_learningrate.epoch[159:],data_learningrate['val_mse_0.001'][159:])
ax.tick_params(axis = 'y',which = 'minor', labelsize = 40)
ax.tick_params(axis = 'y', which = 'major',labelsize = 40)
ax.tick_params(axis = 'x', labelsize = 40)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname = 'Times New Roman')
labels = ax.get_yticklabels(minor = True) + ax.get_yticklabels(minor = False)
for spine in ax.spines.values():
    spine.set_linewidth(2)


fig = plt.figure(figsize = (10,7),dpi=300)
plt.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_decay'][159:])
plt.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_floor'][159:])
plt.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_exp'][159:])
plt.semilogy(data_learningrate.epoch[159:],data_learningrate['val_loss_0.0005'][159:])
plt.semilogy(data_learningrate.epoch[159:],data_learningrate['val_mse_0.003'][159:])
plt.semilogy(data_learningrate.epoch[159:],data_learningrate['val_mse_0.001'][159:])
#plt.legend(['Sc1','Sc2','Sc3','0.0005','0.003','0.001'],fontsize = 15,loc='upper left')
#plt.xlabel('epoch',fontsize = 20)
#plt.ylabel('loss',fontsize = 20)
plt.yticks(fontsize=20)
plt.xticks(fontsize= 20)
plt.show()
fig.savefig('learning rate_accuracy_cut',dpi = 300)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data_activationfunction = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/activation functions.xlsx')

In [ ]:
data_activationfunction

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=300)
plt.semilogy(data_activationfunction.epoch,data_activationfunction['Val_RELU'])
plt.semilogy(data_activationfunction.epoch,data_activationfunction['Val_sigmoid'])
plt.semilogy(data_activationfunction.epoch,data_activationfunction['Val_ELU'])

plt.legend(['RELU','Sigmoid','ELU'],prop = font ,loc='upper left')
plt.xlabel('Training epochs',fontsize = 30,fontname = 'Times New Roman')
plt.ylabel('Validating MSE',fontsize = 30,fontname = 'Times New Roman')
plt.yticks(fontsize=30,fontname = 'Times New Roman')
plt.xticks(fontsize= 30,fontname = 'Times New Roman')
plt.show()
fig.savefig('activationfunction_accuracy',dpi = 300)


In [ ]:
fig, ax = plt.subplots(figsize=(20,15),dpi = 300)
ax.semilogy(data_activationfunction.epoch[159:],data_activationfunction['Val_RELU'][159:],lw = 5)
ax.semilogy(data_activationfunction.epoch[159:],data_activationfunction['Val_sigmoid'][159:],lw = 5)
ax.semilogy(data_activationfunction.epoch[159:],data_activationfunction['Val_ELU'][159:],lw = 5)
ax.tick_params(axis = 'y', which = 'major' , labelsize = 40)
ax.tick_params(axis = 'y',which = 'minor', labelsize = 40)
ax.tick_params(axis= 'x', labelsize = 40)
ax.minorticks_on()
ax.set_xticklabels([int(a) for a in ax.get_xticks()],fontname = 'Times New Roman')
labels = ax.get_yticklabels(minor = True) + ax.get_yticklabels(minor = False)
for label in labels:
    label.set_fontname('Times New Roman')

for spine in ax.spines.values():
    spine.set_linewidth(2.5)

#fig = plt.figure(figsize = (10,7),dpi=300)
#plt.semilogy(data_activationfunction.epoch[159:],data_activationfunction['Val_RELU'][159:])
#plt.semilogy(data_activationfunction.epoch[159:],data_activationfunction['Val_sigmoid'][159:])
#plt.semilogy(data_activationfunction.epoch[159:],data_activationfunction['Val_ELU'][159:])

#plt.legend(['RELU','sigmoid','ELU'],fontsize = 15,loc='upper left')
#plt.xlabel('epoch',fontsize = 20)
#plt.ylabel('loss',fontsize = 20)
#plt.yticks(fontsize=20)
#plt.xticks(fontsize= 20)
plt.show()
fig.savefig('activationfunction_accuracy_cut',dpi = 300)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
data_augment = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/data_augment.xlsx')

In [ ]:
fontPath = '/content/drive/MyDrive/image/times.ttf'
font_manager.fontManager.addfont(fontPath)

In [ ]:
font = FontProperties()
font.set_family('Times New Roman')
font.set_size(20)

In [ ]:
data_augment

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=300)
plt.semilogy(data_augment.epoch,data_augment['val_20%augment'])
plt.semilogy(data_augment.epoch,data_augment['val_20%noaugment'])
plt.legend(['With_augmentation','No_augmentation'],prop=font,loc = 'upper left')
plt.xlabel('Training epochs', fontname='Times New Roman',fontsize = 30)
plt.ylabel('Validating MSE',fontsize = 30,fontname='Times New Roman')
plt.yticks(fontsize=30,fontname='Times New Roman')
plt.xticks(fontsize= 30,fontname='Times New Roman')
fig.savefig('augmentation_accuracy_20%',dpi = 300)

In [ ]:
fig = plt.figure(figsize = (10,7),dpi=300)
plt.semilogy(data_augment.epoch[159:],data_augment['val_20%augment'][159:])
plt.semilogy(data_augment.epoch[159:],data_augment['val_20%noaugment'][159:])
plt.tick_params(axis='y', labelsize=50)
plt.yticks(fontname='Times New Roman')
plt.xticks(fontsize= 30,fontname='Times New Roman')
plt.show()
fig.savefig('augmentation_accuracy_20_cut',dpi = 300)


In [ ]:
fig , ax = plt.subplots(figsize = (15,10), dpi = 300)
ax.semilogy(data_augment.epoch,data_augment['val_20%augment'])
ax.semilogy(data_augment.epoch,data_augment['val_20%noaugment'])
ax.tick_params(axis ='y', which = 'major', labelsize = 30)
ax.tick_params(axis ='y', which = 'minor', labelsize = 30)
ax.tick_params(axis='x', labelsize = 30)
plt.legend(['With_augmentation','No_augmentation'],prop=font,loc = 'upper left')
ax.minorticks_on()
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname='Times New Roman')
labels = ax.get_yticklabels(minor=False) + ax.get_yticklabels(minor=True)
for label in labels:
    label.set_fontname('Times New Roman')

ax.set_xlabel('Training epochs',fontsize = 30 , fontname= 'Times New Roman')
ax.set_ylabel('Validating MSE',fontsize = 30 , fontname= 'Times New Roman')
fig.savefig('augmentation_accuracy_20%',dpi = 300)

In [ ]:
fig, ax = plt.subplots(figsize = (15,9),dpi=300)
ax.semilogy(data_augment.epoch[159:],data_augment['val_20%augment'][159:],lw = 4)
ax.semilogy(data_augment.epoch[159:],data_augment['val_20%noaugment'][159:],lw =4)
ax.tick_params(axis='y',which = 'major', labelsize=40)
ax.tick_params(axis='y',which = 'minor', labelsize=40)
print((ax.get_xticks()))
#ax.set_yticks([0,1], labels=None, minor= True)
# فونت و سایز تیک‌های محور Y (تیک‌های فرعی)
#for tick in ax.yaxis.get_minor_ticks():
    #tick.set_fontsize(40)  # مثلاً 40
    #tick.set_fontname('Times New Roman')
ax.minorticks_on()
# فونت و سایز محور X
ax.tick_params(axis='x', labelsize=40)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname='Times New Roman')
labels = ax.get_yticklabels(minor=False) + ax.get_yticklabels(minor=True)
for label in labels:
    label.set_fontname('Times New Roman')
for spine in ax.spines.values():
    spine.set_linewidth(2)

#ax.set_xlabel('')
#ax.set_ylabel()
# نمایش و ذخیره
plt.tight_layout()

plt.show()
fig.savefig('augmentation_accuracy_20_cut',dpi = 300)

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=300)
plt.semilogy(data_augment.epoch,data_augment['val_50%augment'])
plt.semilogy(data_augment.epoch,data_augment['val_50%noaugment'])
plt.legend(['With_augmentation','No_augmentation'],prop = font ,loc = 'upper left')
plt.xlabel('Training epochs',fontname='Times New Roman',fontsize = 30)
plt.ylabel('Validating MSE',fontname='Times New Roman',fontsize = 30)
plt.yticks(fontsize = 30,fontname='Times New Roman')
plt.xticks(fontsize = 30,fontname='Times New Roman')
fig.savefig('augmentation_accuracy_50%',dpi = 300)

In [ ]:
fig,ax =plt.subplots(figsize = (25,20), dpi = 400)
ax.semilogy(data_augment.epoch[159:],data_augment['val_50%augment'][159:],lw = 5)
ax.semilogy(data_augment.epoch[159:],data_augment['val_50%noaugment'][159:], lw =5)
ax.tick_params(axis='y',which = 'major', labelsize=50)
ax.tick_params(axis='y',which = 'minor', labelsize=50)
ax.minorticks_on()
# فونت و سایز محور X
ax.tick_params(axis='x', labelsize=50)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname='Times New Roman')
ax.set_yticklabels([int(a) for a in ax.get_xticks()], fontname='Times New Roman')

labels = ax.get_yticklabels(minor=False) + ax.get_yticklabels(minor=True)
for label in labels:
    label.set_fontname('Times New Roman')

for spine in ax.spines.values():
    spine.set_linewidth(2.5)

#fig = plt.figure(figsize = (10,7),dpi=300)
#plt.semilogy(data_augment.epoch[159:],data_augment['val_50%augment'][159:])
#plt.semilogy(data_augment.epoch[159:],data_augment['val_50%noaugment'][159:])

#plt.yticks(fontsize=20)
#plt.xticks(fontsize= 30,fontname='Times New Roman')
plt.show()
fig.savefig('augmentation_accuracy50_cut',dpi = 400)

In [ ]:
fig = plt.figure(figsize = (12,9),dpi=300)
plt.semilogy(data_augment.epoch,data_augment['val_all_augment'])
plt.semilogy(data_augment.epoch,data_augment['val_all_noaugment'])
plt.legend(['With_augmentation','No_augmentation'],prop = font,loc = 'upper left')
plt.xlabel('Training epochs',fontsize = 30,fontname = 'Times New Roman')
plt.ylabel('Validating MSE',fontsize = 30,fontname = 'Times New Roman')
plt.yticks(fontsize=30,fontname = 'Times New Roman')
plt.xticks(fontsize= 30,fontname = 'Times New Roman')
fig.savefig('augmentation_accuracy_100%',dpi = 300)

In [ ]:
fig,ax = plt.subplots(figsize = (20,15) , dpi = 300)
ax.semilogy(data_augment.epoch[159:],data_augment['val_all_augment'][159:],lw = 5)
ax.semilogy(data_augment.epoch[159:],data_augment['val_all_noaugment'][159:],lw = 5)
ax.tick_params(axis = 'y', which = 'major', labelsize = 50)
ax.tick_params(axis = 'y', which = 'minor', labelsize = 50)
ax.minorticks_on()
# فونت و سایز محور X
ax.tick_params(axis='x', labelsize=50)
ax.set_xticklabels([int(a) for a in ax.get_xticks()], fontname='Times New Roman')
labels = ax.get_yticklabels(minor=False) + ax.get_yticklabels(minor=True)
for label in labels:
    label.set_fontname('Times New Roman')
for spine in ax.spines.values():
    spine.set_linewidth(2.5)
# نمایش و ذخیره
plt.tight_layout()

plt.show()
fig.savefig('augmentation_accuracy_100_cut',dpi = 300)
#fig = plt.figure(figsize = (10,7),dpi=300)
#plt.semilogy(data_augment.epoch[159:],data_augment['val_all_augment'][159:])
#plt.semilogy(data_augment.epoch[159:],data_augment['val_all_noaugment'][159:])

#plt.yticks(fontsize=20)
#plt.xticks(fontsize= 20)
#plt.show()
#fig.savefig('augmentation_accuracy100_cut',dpi = 300)

In [ ]:
fig = plt.figure(figsize = (17,15),dpi=400)

plt.subplot(2,2,1)
plt.semilogy(data_augment.epoch,data_augment['val_20%augment'])
plt.semilogy(data_augment.epoch,data_augment['val_20%noaugment'])
plt.legend(['20%_with_augment','20%_no_augment'],fontsize = 15)
plt.xlabel('epoch',fontsize = 25)
plt.ylabel('loss',fontsize = 25)
plt.yticks(fontsize=20)
plt.xticks(fontsize= 20)

plt.subplot(2,2,2)
plt.semilogy(data_augment.epoch,data_augment['val_50%augment'])
plt.semilogy(data_augment.epoch,data_augment['val_50%noaugment'])
plt.legend(['50%_with_augment','50%_no_augment'],fontsize = 15)
plt.xlabel('epoch',fontsize = 25)
plt.ylabel('loss',fontsize = 25)
plt.yticks(fontsize=20)
plt.xticks(fontsize= 20)

plt.subplot(2,2,3)

plt.semilogy(data_augment.epoch,data_augment['val_all_augment'])
plt.semilogy(data_augment.epoch,data_augment['val_all_noaugment'])
plt.legend(['100%_with_augment','100%_no_augment'],fontsize = 15)
plt.xlabel('epoch',fontsize = 25)
plt.ylabel('loss',fontsize = 25)
plt.yticks(fontsize=20)
plt.xticks(fontsize= 20)

plt.show()
fig.savefig('augmentation_accuracy',dpi = 400)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = plt.figure(figsize = (10,7),dpi = 200)
x = [1,2]
y = [0.0186,60]
plt.bar(x,y)
plt.xlabel('method',fontsize = 20)
plt.ylabel('Prediction Time(s)',fontsize = 20)
plt.xticks([1,2],['CNN','Maximal ball'],fontsize = 15)
plt.yticks(fontsize = 15)
plt.show()
fig.savefig('wall time',dpi = 200)
